# Model B V1 Training Notebook

This notebook trains the shared multi-head attribute model for StyleSync.

Model purpose:

- take one clothing image
- predict recommendation-friendly attributes
- support `tops`, `bottomwear`, and `outerwear`
- use masked supervision so each category only trains relevant heads

This notebook is MPS-first for Apple Silicon and falls back to CPU automatically.


In [35]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import confusion_matrix, f1_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms


In [36]:
NOTEBOOK_DIR = Path.cwd()
ATTRIBUTE_MODELING_ROOT = NOTEBOOK_DIR.parent
DEEPFASHION_ROOT = ATTRIBUTE_MODELING_ROOT.parent
DATA_ROOT = DEEPFASHION_ROOT / "data"
CSV_PATH = DATA_ROOT / "anno_fine_v2_common_items_material_merged.csv"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Notebook dir: {NOTEBOOK_DIR}")
print(f"DeepFashion root: {DEEPFASHION_ROOT}")
print(f"CSV exists: {CSV_PATH.exists()}")
print(f"Output dir: {OUTPUT_DIR}")


Notebook dir: /Users/jadenwu/Desktop/StyleSync/StyleSync/DeepFashion/attribute_modeling/model_b_v1
DeepFashion root: /Users/jadenwu/Desktop/StyleSync/StyleSync/DeepFashion
CSV exists: True
Output dir: /Users/jadenwu/Desktop/StyleSync/StyleSync/DeepFashion/attribute_modeling/model_b_v1/outputs


In [37]:
SEED = 42
IMAGE_SIZE = 224
RESIZE_SIZE = 256
BATCH_SIZE = 32
NUM_WORKERS = 0
VAL_SIZE = 0.2
USE_OFFICIAL_TEST_AS_HOLDOUT = True
USE_WEIGHTED_MATERIAL_SAMPLER = True
SAMPLER_POWER = 0.6
USE_BBOX_CROP = True
BBOX_CROP_MODE = "padded"
BBOX_PADDING = 0.10

# Two-stage training keeps V2 practical while still letting the backbone adapt.
STAGE1_EPOCHS = 5
STAGE2_EPOCHS = 6
STAGE1_LR = 1e-3
STAGE2_HEAD_LR = 5e-4
STAGE2_BACKBONE_LR = 1e-4
WEIGHT_DECAY = 1.5e-4
DROPOUT = 0.3
LABEL_SMOOTHING = 0.0
EARLY_STOPPING_PATIENCE = 2
SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 2
MAX_CLASS_WEIGHT = 5.0
IGNORE_INDEX = -100
SHARED_DIM = 512

CHECKPOINT_PATH = OUTPUT_DIR / "model_b_v3_pattern_simplified_resnet50_best.pt"
VOCAB_PATH = OUTPUT_DIR / "model_b_v3_pattern_simplified_label_vocabs.json"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")
print(f"Stage 1 epochs: {STAGE1_EPOCHS} | Stage 2 epochs: {STAGE2_EPOCHS}")
print(f"Weighted material sampler enabled: {USE_WEIGHTED_MATERIAL_SAMPLER}")
print(f"Sampler power: {SAMPLER_POWER}")
print(f"Use bbox crop: {USE_BBOX_CROP} | mode: {BBOX_CROP_MODE} | padding: {BBOX_PADDING}")


Using device: mps
Stage 1 epochs: 5 | Stage 2 epochs: 10
Weighted material sampler enabled: True
Sampler power: 0.6
Use bbox crop: True | mode: padded | padding: 0.1


## Head Definitions

The label sets below come directly from the V1 head spec.


In [38]:
HEAD_SPECS = {
    # Each head is its own attribute task with a separate classifier.
    "pattern_family": ["solid", "graphic", "floral", "striped", "other"],
    "material_family": ["denim", "knit", "chiffon", "leather", "other"],
    "sleeve_family": ["sleeveless", "short_sleeve", "long_sleeve"],
}

CATEGORY_HEADS = {
    # This mapping is what turns the model into masked multi-task learning.
    # A head only contributes loss when it is relevant for the current garment category.
    "top": ["pattern_family", "material_family", "sleeve_family"],
    "bottom": ["pattern_family", "material_family"],
    "outerwear": ["pattern_family", "material_family", "sleeve_family"],
}

label_to_idx = {
    head: {label: idx for idx, label in enumerate(labels)}
    for head, labels in HEAD_SPECS.items()
}
idx_to_label = {
    head: {idx: label for idx, label in enumerate(labels)}
    for head, labels in HEAD_SPECS.items()
}

with VOCAB_PATH.open("w") as f:
    json.dump({"head_specs": HEAD_SPECS, "category_heads": CATEGORY_HEADS}, f, indent=2)

HEAD_SPECS


{'pattern_family': ['solid',
  'graphic',
  'floral',
  'striped',
  'embroidered',
  'other'],
 'material_family': ['denim', 'knit', 'chiffon', 'leather', 'other'],
 'sleeve_family': ['sleeveless', 'short_sleeve', 'long_sleeve']}

## Final V3 Label Rationale

This notebook keeps the 3-head recommendation-focused setup, keeps the V2 `faux -> leather` merge, and simplifies `pattern_family` one last time by merging `embroidered` into `other`.

Why `faux` was merged:
- `faux` was too rare as a standalone class.
- `faux` and `leather` overlap visually in this dataset.
- The model was paying a high complexity cost for a small class with weak recall.
- Merging them creates a stronger elevated-material signal, especially for outerwear.

Why `embroidered` was merged into `other`:
- `embroidered` was the weakest pattern class in repeated evaluations.
- It overlaps visually with detail-heavy `other` items in this dataset.
- Removing it simplifies the demo story without changing the 3-head model design.

Why these three heads remain:
- `pattern_family` is the strongest outfit-matching signal.
- `material_family` is noisy, but now more learnable after the merge.
- `sleeve_family` is useful for coverage, layering, and seasonality.

The V3 goal is to keep the best material-friendly image pipeline while cleaning up the weakest remaining pattern class.


In [39]:
df = pd.read_csv(CSV_PATH)
df = df[df["image_exists"] == True].copy()
df = df[df["category_group"].isin(CATEGORY_HEADS.keys())].copy()

for head, labels in HEAD_SPECS.items():
    df[head] = df[head].astype(str)
    if head == "material_family":
        # Defensive fallback in case an older CSV still contains faux labels.
        df[head] = df[head].replace({"faux": "leather"})
    if head == "pattern_family":
        # Final V3 cleanup: collapse the weakest pattern class into the fallback bucket.
        df[head] = df[head].replace({"embroidered": "other"})
    applicable_mask = df["category_group"].map(lambda category: head in CATEGORY_HEADS[category])
    # Non-applicable heads are marked explicitly so the dataset can ignore them later.
    df.loc[~applicable_mask, head] = "not_applicable"
    known_mask = df[head].isin(labels) | (df[head] == "not_applicable")
    unknown_mask = applicable_mask & (~known_mask)
    if unknown_mask.any():
        # Unknown applicable labels are collapsed into a safe fallback bucket.
        print(f"Collapsing {unknown_mask.sum()} rows of unknown {head} labels into 'other'")
        if "other" not in labels:
            raise ValueError(f"Head {head} has unknown labels but no 'other' fallback")
        df.loc[unknown_mask, head] = "other"

print(df.shape)
print(df["category_group"].value_counts().to_string())
display(df[["image_name", "category_group"] + list(HEAD_SPECS.keys())].head())


(11987, 41)
category_group
top          6413
bottom       3294
outerwear    2280


,image_name,category_group,pattern_family,material_family,sleeve_family
0,img/Sweet_Crochet_Blouse/img_00000070.jpg,top,embroidered,chiffon,sleeveless
1,img/Classic_Pencil_Skirt/img_00000010.jpg,bottom,solid,other,not_applicable
2,img/Mid-Rise_-_Acid_Wash_Skinny_Jeans/img_0000...,bottom,solid,denim,not_applicable
3,img/Zippered_Single-Button_Blazer/img_00000078...,outerwear,solid,other,long_sleeve
4,img/Boho_Babe_Crochet_Top/img_00000047.jpg,top,solid,other,long_sleeve


In [40]:
for head, labels in HEAD_SPECS.items():
    applicable_mask = df["category_group"].map(lambda category: head in CATEGORY_HEADS[category])
    df.loc[~applicable_mask, head] = "not_applicable"
    known_mask = df[head].isin(labels) | (df[head] == "not_applicable")
    unknown_mask = applicable_mask & (~known_mask)
    if unknown_mask.any():
        print(f"Collapsing {unknown_mask.sum()} remaining unknown {head} labels into 'other'")
        df.loc[unknown_mask, head] = "other"
    unknown = sorted(set(df.loc[applicable_mask, head].unique()) - set(labels))
    print(head, "unknown labels:", unknown)
    print(df[head].value_counts().to_string())
    print()
    assert len(unknown) == 0, f"Unexpected labels remain in {head}: {unknown}"


pattern_family unknown labels: []
pattern_family
solid          6389
graphic        2275
floral         1497
striped         858
embroidered     608
other           360

material_family unknown labels: []
material_family
other      8027
knit       1426
chiffon    1101
denim       939
leather     494

sleeve_family unknown labels: []
sleeve_family
long_sleeve       4394
not_applicable    3294
short_sleeve      2191
sleeveless        2108



## Split Strategy

Use the official `test` rows as a holdout set and stratify a train/validation split on the remaining rows by `category_group`.


In [41]:
if USE_OFFICIAL_TEST_AS_HOLDOUT and "split" in df.columns:
    train_val_df = df[df["split"] != "test"].copy()
    test_df = df[df["split"] == "test"].copy()
else:
    train_val_df = df.copy()
    test_df = df.iloc[0:0].copy()

train_df, val_df = train_test_split(
    train_val_df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=train_val_df["category_group"],
)

print(f"Train rows: {len(train_df)}")
print(f"Val rows: {len(val_df)}")
print(f"Test rows: {len(test_df)}")
print(train_df["category_group"].value_counts().to_string())

def compute_majority_baselines(frame):
    baselines = {}
    for head in HEAD_SPECS:
        applicable_mask = frame["category_group"].map(lambda category: head in CATEGORY_HEADS[category])
        values = frame.loc[applicable_mask, head].astype(str)
        if len(values) == 0:
            baselines[head] = float("nan")
            continue
        baselines[head] = values.value_counts(normalize=True).iloc[0]
    return baselines

train_majority_baselines = compute_majority_baselines(train_df)
val_majority_baselines = compute_majority_baselines(val_df)
test_majority_baselines = compute_majority_baselines(test_df)

pd.DataFrame({
    "train_majority_baseline": train_majority_baselines,
    "val_majority_baseline": val_majority_baselines,
    "test_majority_baseline": test_majority_baselines,
})


Train rows: 7633
Val rows: 1909
Test rows: 2445
category_group
top          4093
bottom       2094
outerwear    1446


,train_majority_baseline,val_majority_baseline,test_majority_baseline
pattern_family,0.528364,0.550026,0.534151
material_family,0.674571,0.660031,0.661759
sleeve_family,0.501173,0.520202,0.507353


In [42]:
def build_train_transform(image_size=224, resize_size=256):
    # After bbox cropping, keep augmentation non-destructive so subtle pattern/material cues survive.
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.06, hue=0.01),
        transforms.RandomRotation(3),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

def build_eval_transform(image_size=224, resize_size=256):
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

train_transform = build_train_transform(IMAGE_SIZE, RESIZE_SIZE)
eval_transform = build_eval_transform(IMAGE_SIZE, RESIZE_SIZE)

class ModelBAttributeDataset(Dataset):
    def __init__(self, frame, transform, use_bbox_crop=True, bbox_crop_mode="padded", bbox_padding=0.10):
        self.df = frame.reset_index(drop=True)
        self.transform = transform
        self.use_bbox_crop = use_bbox_crop
        self.bbox_crop_mode = bbox_crop_mode
        self.bbox_padding = bbox_padding
        self.bbox_fallback_count = 0

    def __len__(self):
        return len(self.df)

    def _crop_to_bbox(self, image, row):
        width, height = image.size
        x1 = float(row["x1"])
        y1 = float(row["y1"])
        x2 = float(row["x2"])
        y2 = float(row["y2"])

        if self.bbox_crop_mode == "exact":
            left = max(0, min(int(x1), width - 1))
            top = max(0, min(int(y1), height - 1))
            right = max(0, min(int(x2), width))
            bottom = max(0, min(int(y2), height))
        elif self.bbox_crop_mode == "padded":
            box_width = max(x2 - x1, 1.0)
            box_height = max(y2 - y1, 1.0)
            pad_x = box_width * self.bbox_padding
            pad_y = box_height * self.bbox_padding
            left = max(0, min(int(round(x1 - pad_x)), width - 1))
            top = max(0, min(int(round(y1 - pad_y)), height - 1))
            right = max(0, min(int(round(x2 + pad_x)), width))
            bottom = max(0, min(int(round(y2 + pad_y)), height))
        else:
            raise ValueError(f"Unsupported bbox crop mode: {self.bbox_crop_mode}")

        if right <= left or bottom <= top:
            self.bbox_fallback_count += 1
            return image, image.size, True

        cropped = image.crop((left, top, right, bottom))
        return cropped, cropped.size, False

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        original_size = image.size
        crop_size = original_size
        used_bbox_fallback = False

        if self.use_bbox_crop:
            image, crop_size, used_bbox_fallback = self._crop_to_bbox(image, row)

        image = self.transform(image)

        category_group = row["category_group"]
        sample = {
            "image": image,
            "category_group": category_group,
            "image_name": row["image_name"],
            "original_size": original_size,
            "crop_size": crop_size,
            "used_bbox_fallback": used_bbox_fallback,
            "bbox": (int(row["x1"]), int(row["y1"]), int(row["x2"]), int(row["y2"])),
        }

        for head, labels in HEAD_SPECS.items():
            applicable = head in CATEGORY_HEADS[category_group]
            # The mask records whether this sample should supervise the current head.
            sample[f"{head}_mask"] = torch.tensor(applicable, dtype=torch.bool)
            if applicable:
                value = row[head]
                if head == "material_family" and value == "faux":
                    value = "leather"
                if value not in label_to_idx[head]:
                    if "other" in label_to_idx[head]:
                        value = "other"
                    else:
                        raise KeyError(f"Unknown label for {head}: {value}")
                sample[f"{head}_target"] = torch.tensor(label_to_idx[head][value], dtype=torch.long)
            else:
                # IGNORE_INDEX turns off loss and metrics for irrelevant heads.
                sample[f"{head}_target"] = torch.tensor(IGNORE_INDEX, dtype=torch.long)

        return sample


def build_material_sampler(frame, power=0.5):
    material_counts = frame["material_family"].value_counts()
    sample_weights = frame["material_family"].map(
        lambda value: 1.0 / (material_counts[value] ** power)
    ).astype(float)
    sample_weights = sample_weights / sample_weights.mean()
    return WeightedRandomSampler(
        weights=torch.tensor(sample_weights.to_numpy(), dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )


train_dataset = ModelBAttributeDataset(
    train_df,
    transform=train_transform,
    use_bbox_crop=USE_BBOX_CROP,
    bbox_crop_mode=BBOX_CROP_MODE,
    bbox_padding=BBOX_PADDING,
)
val_dataset = ModelBAttributeDataset(
    val_df,
    transform=eval_transform,
    use_bbox_crop=USE_BBOX_CROP,
    bbox_crop_mode=BBOX_CROP_MODE,
    bbox_padding=BBOX_PADDING,
)
test_dataset = ModelBAttributeDataset(
    test_df,
    transform=eval_transform,
    use_bbox_crop=USE_BBOX_CROP,
    bbox_crop_mode=BBOX_CROP_MODE,
    bbox_padding=BBOX_PADDING,
)

train_sampler = build_material_sampler(train_df, power=SAMPLER_POWER) if USE_WEIGHTED_MATERIAL_SAMPLER else None
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=train_sampler is None,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

bbox_preview_df = df[["image_name", "x1", "y1", "x2", "y2"]].head().copy()
display(bbox_preview_df)

bbox_sample = train_dataset[0]
print(batch := bbox_sample["image"].shape)
print("original_size", bbox_sample["original_size"])
print("crop_size", bbox_sample["crop_size"])
print("used_bbox_fallback", bbox_sample["used_bbox_fallback"])
for head in HEAD_SPECS:
    print(head, bbox_sample[f"{head}_target"], bbox_sample[f"{head}_mask"])


,image_name,x1,y1,x2,y2
0,img/Sweet_Crochet_Blouse/img_00000070.jpg,66,75,241,293
1,img/Classic_Pencil_Skirt/img_00000010.jpg,65,88,132,218
2,img/Mid-Rise_-_Acid_Wash_Skinny_Jeans/img_0000...,64,1,129,273
3,img/Zippered_Single-Button_Blazer/img_00000078...,1,12,257,300
4,img/Boho_Babe_Crochet_Top/img_00000047.jpg,50,86,150,249


torch.Size([3, 224, 224])
original_size (240, 300)
crop_size (142, 153)
used_bbox_fallback False
pattern_family tensor(1) tensor(True)
material_family tensor(4) tensor(True)
sleeve_family tensor(2) tensor(True)


## BBox Crop Notes

This notebook can train on the garment region using the DeepFashion bounding boxes, including a lightly padded crop.

Expected effects of bbox cropping:
- `pattern_family` should become less distracted by background pixels
- `material_family` may improve because texture cues are concentrated on the garment
- `sleeve_family` may stay similar or improve slightly because padded crops keep a bit more garment context than exact boxes

When comparing bbox runs to full-image runs, focus on:
- `pattern_family` macro F1
- `material_family` macro F1
- `material_family` specific-to-other rate
- `sleeve_family` macro F1


## Model Definition


In [43]:
class ModelBResNet50(nn.Module):
    def __init__(self, head_specs, shared_dim=512, dropout=0.3, freeze_backbone=True):
        super().__init__()
        weights = models.ResNet50_Weights.IMAGENET1K_V2
        backbone = models.resnet50(weights=weights)
        in_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        # The projection is the shared representation that every head reads from.
        self.projection = nn.Sequential(
            nn.Linear(in_features, shared_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        # ModuleDict is the key multi-head change from a standard single-label ResNet.
        self.heads = nn.ModuleDict({
            head: nn.Linear(shared_dim, len(labels))
            for head, labels in head_specs.items()
        })

        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False

    def forward(self, x):
        # A normal classifier would return one logits tensor.
        # This model returns one logits tensor per head.
        features = self.backbone(x)
        shared = self.projection(features)
        return {head: layer(shared) for head, layer in self.heads.items()}


def freeze_backbone(model):
    for param in model.backbone.parameters():
        param.requires_grad = False


def unfreeze_backbone_layer4(model):
    # V1 only unfreezes the top residual block to limit overfitting risk.
    for param in model.backbone.layer4.parameters():
        param.requires_grad = True


model = ModelBResNet50(HEAD_SPECS, shared_dim=SHARED_DIM, dropout=DROPOUT, freeze_backbone=True).to(device)
model


ModelBResNet50(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
  

In [44]:
def compute_class_weights(frame, max_class_weight=4.0):
    weight_tensors = {}
    for head, labels in HEAD_SPECS.items():
        applicable_mask = frame["category_group"].map(lambda category: head in CATEGORY_HEADS[category])
        values = frame.loc[applicable_mask, head].astype(str)
        counts = values.value_counts()
        total = max(len(values), 1)

        weights = []
        for label in labels:
            count = max(int(counts.get(label, 0)), 1)
            weight = total / (len(labels) * count)
            weights.append(weight)

        weights = np.array(weights, dtype=np.float32)
        weights = np.clip(weights, 1.0, max_class_weight)
        weights = weights / weights.mean()
        weight_tensors[head] = torch.tensor(weights, dtype=torch.float32)

    return weight_tensors


class_weights = compute_class_weights(train_df, max_class_weight=MAX_CLASS_WEIGHT)
for head, weights in class_weights.items():
    print(head, dict(zip(HEAD_SPECS[head], [round(float(x), 3) for x in weights])))

criteria = {
    # Each head gets its own weighted loss, and ignore_index skips non-applicable targets.
    head: nn.CrossEntropyLoss(
        weight=class_weights[head].to(device),
        ignore_index=IGNORE_INDEX,
        label_smoothing=LABEL_SMOOTHING,
    )
    for head in HEAD_SPECS
}

HEAD_LOSS_WEIGHTS = {
    "pattern_family": 1.0,
    "material_family": 1.2,
    "sleeve_family": 0.9,
}


def build_optimizer(model, stage_name):
    if stage_name == "stage1":
        params = [
            {"params": model.projection.parameters(), "lr": STAGE1_LR},
            {"params": model.heads.parameters(), "lr": STAGE1_LR},
        ]
    elif stage_name == "stage2":
        params = [
            {"params": model.backbone.layer4.parameters(), "lr": STAGE2_BACKBONE_LR},
            {"params": model.projection.parameters(), "lr": STAGE2_HEAD_LR},
            {"params": model.heads.parameters(), "lr": STAGE2_HEAD_LR},
        ]
    else:
        raise ValueError(f"Unknown stage: {stage_name}")

    return torch.optim.AdamW(params, weight_decay=WEIGHT_DECAY)


def build_scheduler(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=SCHEDULER_FACTOR,
        patience=SCHEDULER_PATIENCE,
    )


pattern_family {'solid': 0.427, 'graphic': 0.427, 'floral': 0.561, 'striped': 1.037, 'embroidered': 1.414, 'other': 2.135}
material_family {'denim': 1.027, 'knit': 0.689, 'chiffon': 0.876, 'leather': 2.008, 'other': 0.402}
sleeve_family {'sleeveless': 1.116, 'short_sleeve': 1.068, 'long_sleeve': 0.816}


In [45]:
def compute_head_losses(logits_by_head, batch):
    losses = {}
    total_loss = torch.tensor(0.0, device=device)
    for head in HEAD_SPECS:
        targets = batch[f"{head}_target"].to(device)
        logits = logits_by_head[head]
        loss = criteria[head](logits, targets)
        losses[head] = loss
        # Total loss is the sum of the active per-head losses.
        total_loss = total_loss + (HEAD_LOSS_WEIGHTS[head] * loss)
    return total_loss, losses


def init_metric_store():
    return {
        head: {"correct": 0, "total": 0, "targets": [], "preds": []}
        for head in HEAD_SPECS
    }


def update_metrics(metric_store, logits_by_head, batch):
    for head in HEAD_SPECS:
        targets = batch[f"{head}_target"].to(device)
        mask = targets != IGNORE_INDEX
        if mask.sum().item() == 0:
            continue
        preds = logits_by_head[head].argmax(dim=1)
        # Metrics only use applicable targets, so ignored heads never pollute evaluation.
        correct = (preds[mask] == targets[mask]).sum().item()
        total = mask.sum().item()
        metric_store[head]["correct"] += correct
        metric_store[head]["total"] += total
        metric_store[head]["targets"].extend(targets[mask].detach().cpu().tolist())
        metric_store[head]["preds"].extend(preds[mask].detach().cpu().tolist())


def summarize_metrics(metric_store, majority_baselines):
    rows = []
    for head, values in metric_store.items():
        total = values["total"]
        if total == 0:
            accuracy = float("nan")
            macro_f1 = float("nan")
        else:
            accuracy = values["correct"] / total
            macro_f1 = f1_score(values["targets"], values["preds"], average="macro", zero_division=0)
        rows.append({
            "head": head,
            "accuracy": accuracy,
            "majority_baseline": majority_baselines.get(head, float("nan")),
            "macro_f1": macro_f1,
            "support": total,
        })
    return pd.DataFrame(rows)


def summarize_per_class_recall(metric_store):
    rows = []
    for head, values in metric_store.items():
        if values["total"] == 0:
            continue
        recalls = recall_score(
            values["targets"],
            values["preds"],
            labels=list(range(len(HEAD_SPECS[head]))),
            average=None,
            zero_division=0,
        )
        for idx, recall in enumerate(recalls):
            rows.append({
                "head": head,
                "label": HEAD_SPECS[head][idx],
                "recall": recall,
            })
    return pd.DataFrame(rows)


def metrics_to_dict(metrics_df):
    return {
        row["head"]: {
            "accuracy": row["accuracy"],
            "macro_f1": row["macro_f1"],
            "support": row["support"],
        }
        for _, row in metrics_df.iterrows()
    }


In [46]:
def run_epoch(model, loader, majority_baselines, optimizer=None):
    train_mode = optimizer is not None
    if train_mode:
        model.train()
    else:
        model.eval()

    metric_store = init_metric_store()
    total_loss = 0.0
    num_batches = 0

    context = torch.enable_grad() if train_mode else torch.no_grad()
    with context:
        for batch in loader:
            images = batch["image"].to(device)
            # The forward pass returns a dict of logits, one tensor per head.
            logits_by_head = model(images)
            loss, _ = compute_head_losses(logits_by_head, batch)

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            num_batches += 1
            update_metrics(metric_store, logits_by_head, batch)

    avg_loss = total_loss / max(num_batches, 1)
    metrics_df = summarize_metrics(metric_store, majority_baselines)
    return avg_loss, metrics_df, metric_store


def metric_objective(metrics_df):
    return metrics_df["macro_f1"].mean()


def save_checkpoint(model, optimizer, scheduler, history, best_score, best_loss, stage_name, global_epoch):
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "head_specs": HEAD_SPECS,
            "category_heads": CATEGORY_HEADS,
            "history": history,
            "best_score": best_score,
            "best_loss": best_loss,
            "stage_name": stage_name,
            "global_epoch": global_epoch,
            "config": {
                "image_size": IMAGE_SIZE,
                "batch_size": BATCH_SIZE,
                "stage1_epochs": STAGE1_EPOCHS,
                "stage2_epochs": STAGE2_EPOCHS,
                "stage1_lr": STAGE1_LR,
                "stage2_head_lr": STAGE2_HEAD_LR,
                "stage2_backbone_lr": STAGE2_BACKBONE_LR,
                "dropout": DROPOUT,
                "sampler_power": SAMPLER_POWER,
                "label_smoothing": LABEL_SMOOTHING,
            },
        },
        CHECKPOINT_PATH,
    )


history = []
best_val_score = -1.0
best_val_loss = float("inf")
global_epoch = 0


In [47]:
def train_stage(stage_name, epochs):
    global best_val_score, best_val_loss, global_epoch

    if stage_name == "stage1":
        freeze_backbone(model)
    elif stage_name == "stage2":
        freeze_backbone(model)
        unfreeze_backbone_layer4(model)
    else:
        raise ValueError(f"Unknown stage name: {stage_name}")

    optimizer = build_optimizer(model, stage_name)
    scheduler = build_scheduler(optimizer)
    patience_counter = 0

    for stage_epoch in range(epochs):
        global_epoch += 1
        train_loss, train_metrics, _ = run_epoch(model, train_loader, train_majority_baselines, optimizer=optimizer)
        val_loss, val_metrics, val_metric_store = run_epoch(model, val_loader, val_majority_baselines, optimizer=None)
        val_score = metric_objective(val_metrics)
        val_metric_dict = metrics_to_dict(val_metrics)

        history.append({
            "global_epoch": global_epoch,
            "stage": stage_name,
            "stage_epoch": stage_epoch + 1,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "val_macro_f1_mean": val_score,
            "pattern_macro_f1": val_metric_dict["pattern_family"]["macro_f1"],
            "material_macro_f1": val_metric_dict["material_family"]["macro_f1"],
            "sleeve_macro_f1": val_metric_dict["sleeve_family"]["macro_f1"],
        })

        print(f"{stage_name} | epoch {stage_epoch + 1}/{epochs} | global_epoch={global_epoch}")
        print(f"  train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | val_macro_f1_mean={val_score:.4f}")
        print("  validation metrics")
        print(val_metrics.to_string(index=False))

        scheduler.step(val_score)

        improved = (val_score > best_val_score + 1e-4) or (abs(val_score - best_val_score) <= 1e-4 and val_loss < best_val_loss)
        if improved:
            best_val_score = val_score
            best_val_loss = val_loss
            save_checkpoint(model, optimizer, scheduler, history, best_val_score, best_val_loss, stage_name, global_epoch)
            patience_counter = 0
            print(f"  saved checkpoint to {CHECKPOINT_PATH}")
        else:
            patience_counter += 1
            print(f"  no improvement | early stop counter = {patience_counter}/{EARLY_STOPPING_PATIENCE}")
            if patience_counter >= EARLY_STOPPING_PATIENCE:
                print(f"  early stopping triggered for {stage_name}")
                break


train_stage("stage1", STAGE1_EPOCHS)
train_stage("stage2", STAGE2_EPOCHS)


stage1 | epoch 1/5 | global_epoch=1
  train_loss=2.7609 | val_loss=2.4467 | val_macro_f1_mean=0.6450
  validation metrics
           head  accuracy  majority_baseline  macro_f1  support
 pattern_family  0.735464           0.550026  0.559027     1909
material_family  0.592981           0.660031  0.530549     1909
  sleeve_family  0.869408           0.520202  0.845524     1386
  saved checkpoint to /Users/jadenwu/Desktop/StyleSync/StyleSync/DeepFashion/attribute_modeling/model_b_v1/outputs/model_b_v2_material_merged_resnet50_best.pt
stage1 | epoch 2/5 | global_epoch=2
  train_loss=2.1221 | val_loss=2.3357 | val_macro_f1_mean=0.6641
  validation metrics
           head  accuracy  majority_baseline  macro_f1  support
 pattern_family  0.757988           0.550026  0.597586     1909
material_family  0.608696           0.660031  0.548655     1909
  sleeve_family  0.859307           0.520202  0.845976     1386
  saved checkpoint to /Users/jadenwu/Desktop/StyleSync/StyleSync/DeepFashion/attribut

In [48]:
history_df = pd.DataFrame(history)
display(history_df)

best_overall_row = history_df.loc[history_df["val_macro_f1_mean"].idxmax()]
best_pattern_row = history_df.loc[history_df["pattern_macro_f1"].idxmax()]
best_material_row = history_df.loc[history_df["material_macro_f1"].idxmax()]

print("Best overall checkpoint row:")
display(best_overall_row.to_frame().T)
print("Best pattern row:")
display(best_pattern_row.to_frame().T)
print("Best material row:")
display(best_material_row.to_frame().T)


,global_epoch,stage,stage_epoch,train_loss,val_loss,val_macro_f1_mean,pattern_macro_f1,material_macro_f1,sleeve_macro_f1
0,1,stage1,1,2.760945,2.446677,0.645033,0.559027,0.530549,0.845524
1,2,stage1,2,2.122111,2.335730,0.664073,0.597586,0.548655,0.845976
2,3,stage1,3,1.812650,2.492169,0.652402,0.599340,0.503707,0.854160
3,4,stage1,4,1.625000,2.335686,0.671793,0.598010,0.555947,0.861421
4,5,stage1,5,1.507706,2.304842,0.674374,0.579523,0.581956,0.861642
5,6,stage2,1,1.200992,2.212587,0.707980,0.626034,0.613568,0.884337
6,7,stage2,2,0.837685,2.275804,0.716269,0.635544,0.623176,0.890088
7,8,stage2,3,0.625312,2.329429,0.736416,0.652830,0.661214,0.895202
8,9,stage2,4,0.515521,2.616594,0.720929,0.620381,0.641871,0.900536
9,10,stage2,5,0.403096,2.657386,0.723332,0.618788,0.651950,0.899260


Best overall checkpoint row:


,global_epoch,stage,stage_epoch,train_loss,val_loss,val_macro_f1_mean,pattern_macro_f1,material_macro_f1,sleeve_macro_f1
7,8,stage2,3,0.625312,2.329429,0.736416,0.65283,0.661214,0.895202


Best pattern row:


,global_epoch,stage,stage_epoch,train_loss,val_loss,val_macro_f1_mean,pattern_macro_f1,material_macro_f1,sleeve_macro_f1
7,8,stage2,3,0.625312,2.329429,0.736416,0.65283,0.661214,0.895202


Best material row:


,global_epoch,stage,stage_epoch,train_loss,val_loss,val_macro_f1_mean,pattern_macro_f1,material_macro_f1,sleeve_macro_f1
7,8,stage2,3,0.625312,2.329429,0.736416,0.65283,0.661214,0.895202


## Why The Baseline Overfit

In the earlier baseline run, training loss dropped steadily while validation loss stopped improving after the first few epochs. That pattern suggests the model was fitting the training distribution faster than it was learning reusable features.

The likely causes were:
- no train-time augmentation
- a fully frozen backbone for the entire run
- class imbalance, especially on `material_family`
- checkpointing on loss alone instead of a more task-aware metric
- a weak standalone `faux` class that split the elevated-material signal too thin

This V2 notebook addresses that by adding moderate augmentation, weighted losses, optional weighted sampling, partial stage-two unfreezing, a `faux -> leather` merge, macro-F1 tracking, and early stopping.


## Why Accuracy Alone Was Misleading

Raw accuracy is not enough for this model because some heads have strong majority-class baselines. `material_family`, for example, can look superficially good by overpredicting `other`.

That is why this notebook now reports:
- majority-class baseline per head
- macro F1 per head
- per-class recall per head

For the V2 material schema, success means the merged `leather` class becomes more learnable while `other` becomes less dominant as a fallback prediction.


In [49]:
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])

test_loss, test_metrics, test_metric_store = run_epoch(model, test_loader, test_majority_baselines, optimizer=None)
print(f"Loaded best checkpoint from stage: {checkpoint['stage_name']} | global_epoch={checkpoint['global_epoch']}")
print(f"Test loss: {test_loss:.4f}")
test_metrics


Loaded best checkpoint from stage: stage2 | global_epoch=8
Test loss: 2.5034


,head,accuracy,majority_baseline,macro_f1,support
0,pattern_family,0.784458,0.534151,0.670429,2445
1,material_family,0.691616,0.661759,0.615111,2445
2,sleeve_family,0.910068,0.507353,0.901008,1768


In [50]:
test_recall_df = summarize_per_class_recall(test_metric_store)
display(test_recall_df)

for head in HEAD_SPECS:
    y_true = test_metric_store[head]["targets"]
    y_pred = test_metric_store[head]["preds"]
    if len(y_true) == 0:
        continue
    labels = list(range(len(HEAD_SPECS[head])))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    print(f"\nConfusion matrix for {head}")
    print(pd.DataFrame(cm, index=HEAD_SPECS[head], columns=HEAD_SPECS[head]))

material_true = np.array(test_metric_store["material_family"]["targets"])
material_pred = np.array(test_metric_store["material_family"]["preds"])
material_labels = HEAD_SPECS["material_family"]
other_idx = material_labels.index("other")

true_specific_mask = material_true != other_idx
true_other_mask = material_true == other_idx
pred_other_mask = material_pred == other_idx
pred_specific_mask = material_pred != other_idx

specific_to_other = float((pred_other_mask[true_specific_mask]).mean()) if true_specific_mask.any() else float("nan")
other_to_specific = float((pred_specific_mask[true_other_mask]).mean()) if true_other_mask.any() else float("nan")

material_other_analysis = pd.DataFrame([
    {
        "metric": "true_specific_predicted_other",
        "value": specific_to_other,
    },
    {
        "metric": "true_other_predicted_specific",
        "value": other_to_specific,
    },
    {
        "metric": "specific_to_other_count",
        "value": int(((true_specific_mask) & (pred_other_mask)).sum()),
    },
    {
        "metric": "other_to_specific_count",
        "value": int(((true_other_mask) & (pred_specific_mask)).sum()),
    },
])

print("\nMaterial-family other analysis")
display(material_other_analysis)


,head,label,recall
0,pattern_family,solid,0.899694
1,pattern_family,graphic,0.645089
2,pattern_family,floral,0.715753
3,pattern_family,striped,0.848168
4,pattern_family,embroidered,0.333333
5,pattern_family,other,0.528571
6,material_family,denim,0.814433
7,material_family,knit,0.698305
8,material_family,chiffon,0.538462
9,material_family,leather,0.432692



Confusion matrix for pattern_family
             solid  graphic  floral  striped  embroidered  other
solid         1175       31      38       15           30     17
graphic         53      289      84        7            9      6
floral          34       40     209        2            5      2
striped         22        3       1      162            2      1
embroidered     60       14      15        2           46      1
other           19        7       1        5            1     37

Confusion matrix for material_family
         denim  knit  chiffon  leather  other
denim      158     3        3        0     30
knit         1   206       30        2     56
chiffon      0    18      126        2     88
leather      6    19        9       45     25
other       28   141      266       27   1156

Confusion matrix for sleeve_family
              sleeveless  short_sleeve  long_sleeve
sleeveless           395            25           22
short_sleeve          18           370           41
lo

,metric,value
0,true_specific_predicted_other,0.240629
1,true_other_predicted_specific,0.285538
2,specific_to_other_count,199.000000
3,other_to_specific_count,462.000000


In [51]:
def inspect_predictions(model, dataset, n=8):
    model.eval()
    rows = []
    with torch.no_grad():
        for idx in range(min(n, len(dataset))):
            sample = dataset[idx]
            image = sample["image"].unsqueeze(0).to(device)
            logits = model(image)
            row = {
                "image_name": sample["image_name"],
                "category_group": sample["category_group"],
            }
            for head in HEAD_SPECS:
                if not sample[f"{head}_mask"].item():
                    row[f"{head}_true"] = "ignored"
                    row[f"{head}_pred"] = "ignored"
                    continue
                true_idx = sample[f"{head}_target"].item()
                pred_idx = logits[head].argmax(dim=1).item()
                row[f"{head}_true"] = idx_to_label[head][true_idx]
                row[f"{head}_pred"] = idx_to_label[head][pred_idx]
            rows.append(row)
    return pd.DataFrame(rows)


inspect_predictions(model, val_dataset, n=8)


,image_name,category_group,pattern_family_true,pattern_family_pred,material_family_true,material_family_pred,sleeve_family_true,sleeve_family_pred
0,img/Athletic-Striped_Pencil_Skirt/img_00000011...,bottom,striped,striped,other,other,ignored,ignored
1,img/Bout_That_Graphic_Crop_Top/img_00000002.jpg,top,solid,embroidered,other,knit,sleeveless,long_sleeve
2,img/Pleated_Textured_Skirt/img_00000025.jpg,bottom,other,solid,other,leather,ignored,ignored
3,img/Las_Vegas_81_Tee/img_00000004.jpg,top,solid,solid,other,other,short_sleeve,sleeveless
4,img/Crocheted_Tank/img_00000056.jpg,top,solid,embroidered,other,chiffon,sleeveless,sleeveless
5,img/Cuffed-Sleeve_Striped_Tee/img_00000006.jpg,top,solid,solid,other,other,short_sleeve,short_sleeve
6,img/LA_Clippers_Graphic_Tank/img_00000040.jpg,top,graphic,graphic,other,other,sleeveless,sleeveless
7,img/Embroidered_Mesh-Paneled_Blouse/img_000000...,top,embroidered,embroidered,other,chiffon,sleeveless,sleeveless


## Optional Next Experiments

If the balanced V2 bbox notebook still underperforms, the next most likely improvements are:
- comparing 10% padded bbox cropping against exact bbox cropping if rare-class gains are unclear
- tuning `SAMPLER_POWER` between `0.25` and `0.75`
- trying `material_family` loss weights between `1.1` and `1.3`
- testing whether stage 2 beyond 10 epochs still helps once early stopping is active
- trying a lighter backbone like `resnet18` for faster iteration
- moving from notebook-only experimentation into a reusable training script once the recipe stabilizes
